# GUIDO Clinical Risk Engine: Cloud Master Pipeline (5 TB Google Drive)
---
**Clinical Recurrence & Progression Risk Prediction System (Apollo & CDSCO SaMD Ready)**

### Architectural Highlights:
1. **Zero Local Footprint**: Streams multi-gigabyte genomic cohorts directly into 5 TB Google Drive at datacenter speeds (1 Gbps+).
2. **Sharded Apache Parquet**: Bypasses Google Drive FUSE latency and rate limits by packaging matrices into compressed binary shards.
3. **Standardized NIH TCGA-CDR Outcomes**: Uses curated Progression-Free Interval (PFI) and Overall Survival (OS).
4. **12 Canonical Cancer Hallmarks**: Leak-free metagene pathway construction (Proliferation, EMT, Angiogenesis, Hypoxia, Warburg Glycolysis, Immune Exhaustion, etc.).
5. **Stacking Ensemble (StacK-Risk)**: ElasticNet + Regularized XGBoost + ExtraTrees + Calibrated Meta-Learner with IPCW censoring weights.
6. **Independent Hospital Benchmarks**: Automated out-of-distribution validation against METABRIC (breast cancer) and GEO GSE31210 (lung cancer).

In [ ]:
# Cell 1: Mount Google Drive & Environment Verification
from google.colab import drive
import os, sys
from pathlib import Path

drive.mount('/content/drive')

# Define root storage path on 5 TB Google Drive
DRIVE_ROOT = Path('/content/drive/MyDrive/GUIDO_5TB_DATA')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print(f'[OK] 5 TB Google Drive Root Initialized: {DRIVE_ROOT}')

# Clone or update GUIDO repository in Colab
if not os.path.exists('/content/guido'):
    !git clone https://github.com/Madhavkannank/guido.git /content/guido
%cd /content/guido
!git checkout main && git pull origin main

# Link data/models/results to persistent 5 TB Google Drive
!mkdir -p /content/drive/MyDrive/GUIDO_5TB_DATA/data/raw/xena
!mkdir -p /content/drive/MyDrive/GUIDO_5TB_DATA/models_artifacts
!mkdir -p /content/drive/MyDrive/GUIDO_5TB_DATA/results

!rm -rf data/raw/xena models_artifacts results
!ln -s /content/drive/MyDrive/GUIDO_5TB_DATA/data/raw/xena data/raw/xena
!ln -s /content/drive/MyDrive/GUIDO_5TB_DATA/models_artifacts models_artifacts
!ln -s /content/drive/MyDrive/GUIDO_5TB_DATA/results results
print('[OK] Persistent Drive Symlinks Established (Zero Local VM Footprint)')

# GPU Acceleration Verification
try:
    import torch
    print(f'CUDA Available: {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        print(f'Allocated GPU: {torch.cuda.get_device_name(0)}')
except ImportError:
    print('PyTorch not loaded yet.')

In [ ]:
# Cell 2: Install High-Performance Bio-ML Stack
!pip install -q xgboost scikit-learn joblib pyarrow fastparquet scipy pandas numpy requests lifelines

## Phase 1: High-Speed Direct Ingestion to 5 TB Google Drive
Streams expression matrices and standardized clinical survival tables from official UCSC Xena CDN endpoints directly into Drive without touching local disk.

In [ ]:
# Cell 3: Direct Cloud Ingestion Engine
import urllib.request, gzip, shutil

RAW_DIR = DRIVE_ROOT / 'data' / 'raw' / 'xena'
RAW_DIR.mkdir(parents=True, exist_ok=True)

USER_AGENT = 'GUIDO-Cloud-Pipeline/1.0 (Mozilla/5.0)'
TCGA_CDR_URL = 'https://pancanatlas.xenahubs.net/download/Survival_SupplementalTable_S1_20171025_xena_sp'
BASE_EXPR_URL = 'https://tcga.xenahubs.net/download/TCGA.{cancer}.sampleMap%2FHiSeqV2.gz'

def download_to_drive(url, dest_path, is_gz=False):
    if dest_path.exists() and dest_path.stat().st_size > 1000:
        print(f'[EXISTS] {dest_path.name} ({dest_path.stat().st_size} bytes)')
        return True
    print(f'Downloading {url} -> {dest_path.name} ...')
    req = urllib.request.Request(url, headers={'User-Agent': USER_AGENT})
    if is_gz:
        tmp_gz = RAW_DIR / f'temp_{dest_path.name}.gz'
        with urllib.request.urlopen(req, timeout=120) as resp, open(tmp_gz, 'wb') as f_out:
            shutil.copyfileobj(resp, f_out)
        print(f'Decompressing {dest_path.name} ...')
        with gzip.open(tmp_gz, 'rb') as f_in, open(dest_path, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
        if tmp_gz.exists(): tmp_gz.unlink()
    else:
        with urllib.request.urlopen(req, timeout=120) as resp, open(dest_path, 'wb') as f_out:
            shutil.copyfileobj(resp, f_out)
    print(f'[SAVED] {dest_path.name} ({dest_path.stat().st_size} bytes)')
    return True

# 1. Fetch unified Pan-Cancer TCGA-CDR survival matrix
download_to_drive(TCGA_CDR_URL, RAW_DIR / 'TCGA_CDR_survival.tsv', is_gz=False)

# 2. Fetch target cohorts
TARGET_COHORTS = ['KIRC', 'KIRP', 'KICH', 'LUAD', 'LUSC', 'BRCA', 'COAD', 'STAD', 'LIHC']
for c in TARGET_COHORTS:
    download_to_drive(BASE_EXPR_URL.format(cancer=c), RAW_DIR / f'TCGA.{c}.sampleMap_HiSeqV2.tsv', is_gz=True)
print('[DONE] All target cohorts stored in 5 TB Google Drive.')

## Phase 2: Sharded Apache Parquet Conversion
Converts raw TSV files into compressed Parquet shards (snappy/zstd) to maximize throughput and eliminate Google Drive FUSE latency.

In [ ]:
# Cell 4: Parquet Conversion
import pandas as pd

PARQUET_DIR = DRIVE_ROOT / 'data' / 'processed' / 'shards'
PARQUET_DIR.mkdir(parents=True, exist_ok=True)

for c in TARGET_COHORTS:
    tsv_file = RAW_DIR / f'TCGA.{c}.sampleMap_HiSeqV2.tsv'
    pq_file = PARQUET_DIR / f'TCGA_{c}_expression.parquet'
    if tsv_file.exists() and not pq_file.exists():
        print(f'Converting {tsv_file.name} -> {pq_file.name} ...')
        df = pd.read_csv(tsv_file, sep='\t', index_col=0, low_memory=False)
        df.to_parquet(pq_file, compression='snappy')
        print(f'[PARQUET SAVED] {pq_file.name} ({pq_file.stat().st_size} bytes)')
print('[DONE] Parquet shard conversion complete.')

## Phase 3: Train Clinical Genomic Risk Engine (StacK-Risk)
Trains ElasticNet + XGBoost + ExtraTrees stacking ensembles with 12 Hallmark Metagenes and IPCW censoring correction.

In [ ]:
# Cell 5: Train Clinical Risk Models on Pan-Organ Cohorts
# Point model saving directory to 5 TB Google Drive
MODELS_DIR = DRIVE_ROOT / 'models_artifacts'
RESULTS_DIR = DRIVE_ROOT / 'results'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'[OK] Model artifacts target directory: {MODELS_DIR}')
print(f'[OK] Validation reports target directory: {RESULTS_DIR}')

# Run retrain_fixed.py across defined pan-organ cohorts
!python retrain_fixed.py PAN-RENAL PAN-LUNG PAN-GI PAN-BREAST

## Phase 4: Independent External Hospital Validation
Evaluates models out-of-distribution against METABRIC (breast cancer) and GEO GSE31210 (lung cancer) without fine-tuning.

In [ ]:
# Cell 6: External Validation Runner
print('[INFO] Executing out-of-distribution benchmarks against independent cohorts...')
!python scripts/run_cross_cohort_validation.py --model TCGA-PAN-RENAL --dataset synthetic
!python scripts/run_cross_cohort_validation.py --model TCGA-PAN-BREAST --dataset metabric
print('[DONE] Validation reports saved to Google Drive.')